# ego2robot: train SmolVLA on robot data generated from my phone demos

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells one by one (Shift+Enter).

Pipeline: 20 phone demos → retargeted to a Panda in simulation (`sim/replay_demos.py --save`, run on the Mac)
→ LeRobot dataset → fine-tune `lerobot/smolvla_base` → evaluate on 50 unseen random scenes.

## 1. Settings: change the first line to your GitHub username

In [ ]:
GITHUB_USER = "YOUR_GITHUB_USERNAME"   # <- change this
BRANCH = "train-smolvla"
EPISODES_ZIP = "/content/drive/MyDrive/ego2robot/sim_episodes.zip"   # uploaded from the Mac
WORK = "/content/drive/MyDrive/ego2robot"                            # checkpoints + results are saved here (survive disconnects)
STEPS, BATCH = 6000, 16

## 2. Connect Google Drive and download the code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!rm -rf /content/ego2robot && git clone -q -b {BRANCH} https://github.com/{GITHUB_USER}/ego2robot.git /content/ego2robot
%cd /content/ego2robot
!git log --oneline -3

## 3. Install LeRobot (SmolVLA) and the simulator (~3 min)

In [ ]:
!pip install -q "lerobot[smolvla,dataset]==0.6.1" "robosuite==1.4.0" "mujoco==3.8.1" termcolor numba "imageio[ffmpeg]"
import os
os.environ["MUJOCO_GL"] = "egl"; os.environ["PYOPENGL_PLATFORM"] = "egl"   # GPU rendering for the evaluation
!python -c "import lerobot, robosuite, torch; print('lerobot', lerobot.__version__, '| robosuite', robosuite.__version__, '| cuda', torch.cuda.is_available())"

## 4. Unpack the simulated episodes and convert them to a LeRobot dataset (~10-15 min)

In [ ]:
!mkdir -p /content/sim_episodes && unzip -q -o {EPISODES_ZIP} -d /content/sim_episodes
EP_DIR = "/content/sim_episodes/sim_episodes"   # the zip contains a folder called sim_episodes
!ls {EP_DIR} | grep -c "npz$"
!rm -rf /content/ego2robot_ds && python -m sim.to_lerobot {EP_DIR} --repo-id local/ego2robot_bowl_on_plate --root /content/ego2robot_ds 2>&1 | grep -v "moov atom" | tail -3

## 5. Fine-tune SmolVLA (~2-3 h on a T4)

Checkpoints are written to Google Drive every 1000 steps. If Colab disconnects, re-run cells 1-4,
then run this cell again with `RESUME = True`.

In [ ]:
RESUME = False
OUT = f"{WORK}/train/smolvla"
if RESUME:
    cmd = f"lerobot-train --config_path={OUT}/checkpoints/last/pretrained_model/train_config.json --resume=true"
else:
    !rm -rf {OUT}
    cmd = " ".join([
        "lerobot-train",
        "--policy.path=lerobot/smolvla_base",
        "--policy.push_to_hub=false",
        "--policy.device=cuda",
        "--dataset.repo_id=local/ego2robot_bowl_on_plate",
        "--dataset.root=/content/ego2robot_ds",
        f"--batch_size={BATCH}", f"--steps={STEPS}",
        "--save_freq=1000", "--log_freq=100", "--num_workers=2",
        f"--output_dir={OUT}", "--job_name=smolvla_ego2robot",
        "--wandb.enable=false",
    ])
print(cmd)
!{cmd}

## 6. Evaluate on 50 random scenes the policy has never seen (~20 min)

In [ ]:
CKPT = f"{WORK}/train/smolvla/checkpoints/last/pretrained_model"
!python -m sim.eval_policy {CKPT} --episodes 50 --videos 6 --out {WORK}/eval_smolvla 2>&1 | grep -v WARNING

Results and the first 6 videos are now in Google Drive → `ego2robot/eval_smolvla/`.